# Notebook 05 — Enriquecimento da Camada Gold

## Objetivos

Neste notebook realizamos o enriquecimento analítico da camada Gold:

1. **Verificar integridade referencial** entre `fact_vendas` e todas as dimensões
2. **Calcular agregações de negócio** usando joins com as dimensões
3. **Criar tabelas agregadas** na camada Gold para consumo direto por BI

As agregações produzidas são: receita por estado, receita por categoria, tendência mensal e ranking de vendedores.


## 1. Criação da SparkSession


In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, round as spark_round, avg, desc, asc, month as spark_month, year as spark_year

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
DATA_DIR = os.path.join(PROJECT_ROOT, "data")
GOLD_DIR = os.path.join(DATA_DIR, "gold")

spark = (
    SparkSession.builder
    .appName("NB05_Gold_Enriquecimento")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .master("local[*]")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"SparkSession iniciada. Versão: {spark.version}")


## 2. Leitura das Tabelas Gold

Carregamos a tabela fato e todas as dimensões.


In [ ]:
df_fact = spark.read.format("delta").load(os.path.join(GOLD_DIR, "fact_vendas"))
df_dim_clientes = spark.read.format("delta").load(os.path.join(GOLD_DIR, "dim_clientes"))
df_dim_produtos = spark.read.format("delta").load(os.path.join(GOLD_DIR, "dim_produtos"))
df_dim_calendario = spark.read.format("delta").load(os.path.join(GOLD_DIR, "dim_calendario"))
df_dim_vendedores = spark.read.format("delta").load(os.path.join(GOLD_DIR, "dim_vendedores"))

print("Tabelas Gold carregadas:")
print(f"  fact_vendas:       {df_fact.count()} registros")
print(f"  dim_clientes:      {df_dim_clientes.count()} registros")
print(f"  dim_produtos:      {df_dim_produtos.count()} registros")
print(f"  dim_calendario:    {df_dim_calendario.count()} registros")
print(f"  dim_vendedores:    {df_dim_vendedores.count()} registros")


## 3. Verificação de Integridade Referencial

Verificamos se todas as FK em `fact_vendas` têm correspondência nas dimensões.


In [ ]:
print("=" * 60)
print("INTEGRIDADE REFERENCIAL — fact_vendas x Dimensões")
print("=" * 60)

dim_clientes_ids = [r["id_cliente"] for r in df_dim_clientes.select("id_cliente").distinct().collect()]
dim_produtos_ids = [r["id_produto"] for r in df_dim_produtos.select("id_produto").distinct().collect()]
dim_calendario_datas = [r["data"] for r in df_dim_calendario.select("data").distinct().collect()]
dim_vendedores_ids = [r["id_vendedor"] for r in df_dim_vendedores.select("id_vendedor").distinct().collect()]

total_fact = df_fact.count()

missing_clientes = df_fact.filter(~col("id_cliente").isin(dim_clientes_ids)).count()
missing_produtos = df_fact.filter(~col("id_produto").isin(dim_produtos_ids)).count()
missing_datas = df_fact.filter(~col("data_pedido").isin(dim_calendario_datas)).count()
missing_vendedores = df_fact.filter(~col("id_vendedor").isin(dim_vendedores_ids)).count()

print(f"  fact_vendas.id_cliente -> dim_clientes:   {missing_clientes} órfãos")
print(f"  fact_vendas.id_produto -> dim_produtos:   {missing_produtos} órfãos")
print(f"  fact_vendas.data_pedido -> dim_calendario: {missing_datas} órfãos")
print(f"  fact_vendas.id_vendedor -> dim_vendedores: {missing_vendedores} órfãos")

all_ok = (missing_clientes + missing_produtos + missing_datas + missing_vendedores) == 0
print(f"\nIntegridade referencial: {'OK' if all_ok else 'FALHA'}")


## 4. Agregações de Negócio

Calculamos as principais métricas usando joins entre `fact_vendas` e as dimensões.


### 4.1 Receita Total por Estado


In [ ]:
df_receita_estado = df_fact \
    .groupBy("estado") \
    .agg(
        spark_sum("total_pedido").alias("receita_total"),
        count("pedido_id").alias("qtd_pedidos"),
        spark_round(avg("total_pedido"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("receita_total"))

print("Receita Total por Estado:")
df_receita_estado.show(27, truncate=False)


### 4.2 Receita Total por Categoria


In [ ]:
df_receita_categoria = df_fact \
    .join(df_dim_produtos, on="id_produto", how="inner") \
    .groupBy("categoria") \
    .agg(
        spark_sum("total_pedido").alias("receita_total"),
        count("pedido_id").alias("qtd_pedidos"),
        spark_round(avg("total_pedido"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("receita_total"))

print("Receita Total por Categoria:")
df_receita_categoria.show(10, truncate=False)


### 4.3 Tendência de Receita Mensal


In [ ]:
df_receita_mensal = df_fact \
    .join(df_dim_calendario, df_fact["data_pedido"] == df_dim_calendario["data"], how="inner") \
    .groupBy("ano", "mes_num", "nome_mes") \
    .agg(
        spark_sum("total_pedido").alias("receita_total"),
        count("pedido_id").alias("qtd_pedidos")
    ) \
    .orderBy("ano", "mes_num")

print("Tendência de Receita Mensal:")
df_receita_mensal.show(24, truncate=False)


### 4.4 Ranking de Vendedores


In [ ]:
df_ranking_vendedores = df_fact \
    .join(df_dim_vendedores, on="id_vendedor", how="inner") \
    .groupBy("id_vendedor", "nome", "regiao") \
    .agg(
        spark_sum("total_pedido").alias("receita_total"),
        count("pedido_id").alias("qtd_pedidos"),
        spark_round(avg("total_pedido"), 2).alias("ticket_medio")
    ) \
    .orderBy(desc("receita_total"))

print("Ranking de Vendedores:")
df_ranking_vendedores.show(15, truncate=False)


## 5. Criação de Tabelas Agregadas na Gold

Salvamos as agregações como tabelas Delta para consumo direto por ferramentas de BI.


In [ ]:
gold_agg_estado_path = os.path.join(GOLD_DIR, "gold_agg_vendas_estado")
df_receita_estado.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(gold_agg_estado_path)
print(f"[OK] gold_agg_vendas_estado salvo")

gold_agg_mensal_path = os.path.join(GOLD_DIR, "gold_agg_vendas_mensal")
df_receita_mensal.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(gold_agg_mensal_path)
print(f"[OK] gold_agg_vendas_mensal salvo")

gold_agg_vendedor_path = os.path.join(GOLD_DIR, "gold_agg_vendas_vendedor")
df_ranking_vendedores.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(gold_agg_vendedor_path)
print(f"[OK] gold_agg_vendas_vendedor salvo")


## 6. Encerramento da SparkSession


In [ ]:
spark.stop()
print("SparkSession encerrada.")


## Resumo do Enriquecimento

Agregações criadas na camada Gold:

| Tabela Agregada | Descrição | Granularidade |
|----------------|-------------|---------------|
| `gold_agg_vendas_estado` | Receita, pedidos e ticket médio | Estado |
| `gold_agg_vendas_mensal` | Tendência mensal de receita | Mês |
| `gold_agg_vendas_vendedor` | Performance dos vendedores | Vendedor |

Estas tabelas estão prontas para serem conectadas a ferramentas de BI como Power BI, Tableau ou Metabase.

No próximo notebook (**NB06**), realizaremos a **validação final** de qualidade dos dados.
